In [17]:
from pathlib import Path
import pandas as pd
from lxml import etree
import yaml
from pathlib import Path
from lxml import etree
import pandas as pd
from pathlib import Path
import yaml
import re

# Flatten all substances

In [188]:
def extract_substances_to_excel(xml_folder, output_excel, sheet):
    
    ns = {'ipc': 'http://webstds.ipc.org/175x/2.0'}
    substances = set()
    xml_folder = Path(xml_folder)

    for xml_file in xml_folder.glob("*.xml"):
        try:
            tree = etree.parse(xml_file)
            root = tree.getroot()

            for sub in root.findall(".//ipc:Substance", ns):
                name = sub.get("name")
                if name:
                    substances.add(name.strip())

        except Exception as e:
            print(f"Error processing {xml_file.name}: {e}")

    df = pd.DataFrame(sorted(substances), columns=["substance name"])
    df["ecoinvent activity"] = ""
    df["location"] = ""

    with pd.ExcelWriter(output_excel, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
        df.to_excel(writer, sheet_name=sheet, index=False)

    print(f"{len(df)} unique substances written to sheet '{sheet}' in {output_excel}")

In [189]:
extract_substances_to_excel(xml_folder = "data/raw/all", output_excel="data/mapping_all.xlsx", sheet = "Sheet2")

76 unique substances written to sheet 'Sheet2' in data/mapping_all.xlsx


# Flatten all subproducts

In [226]:
def extract_subproducts_to_excel(xml_folder, output_excel, sheet):
    
    ns = {'ipc': 'http://webstds.ipc.org/175x/2.0'}
    subproducts = set()
    xml_folder = Path(xml_folder)

    for xml_file in xml_folder.glob("*.xml"):
    #print(xml_file)

        try:
            tree = etree.parse(xml_file)
            root = tree.getroot()
    
            for sub in root.findall(".//ipc:SubProduct/ipc:ProductID", ns):
                name = sub.get("itemName")
    
                if name:
                    subproducts.add(name.strip())
    
        except Exception as e:
            print(f"Error processing {xml_file.name}: {e}")

        

    df = pd.DataFrame(sorted(subproducts), columns=["subproduct name"])
    df["ecoinvent activity"] = ""
    df["location"] = ""

    with pd.ExcelWriter(output_excel, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
        df.to_excel(writer, sheet_name=sheet, index=False)

    print(f"{len(df)} unique subproducts written to sheet '{sheet}' in {output_excel}")

In [229]:
extract_subproducts_to_excel(xml_folder = "data/raw/all", output_excel="data/mapping_all.xlsx", sheet = "Sheet3")

14 unique subproducts written to sheet 'Sheet3' in data/mapping_all.xlsx


# Flatten all homogeneous materials

In [18]:
def extract_HomogeneousMaterial_to_excel(xml_folder, output_excel, sheet):
    
    ns = {'ipc': 'http://webstds.ipc.org/175x/2.0'}
    HomogeneousMaterial = set()
    xml_folder = Path(xml_folder)

    for xml_file in xml_folder.glob("*.xml"):
    #print(xml_file)

        try:
            tree = etree.parse(xml_file)
            root = tree.getroot()
    
            for sub in root.findall(".//ipc:HomogeneousMaterial", ns):
                name = sub.get("name")
    
                if name:
                    HomogeneousMaterial.add(name.strip())
    
        except Exception as e:
            print(f"Error processing {xml_file.name}: {e}")

        

    df = pd.DataFrame(sorted(HomogeneousMaterial), columns=["homogeneous material"])
    df["ecoinvent activity"] = ""
    df["location"] = ""

    with pd.ExcelWriter(output_excel, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
        df.to_excel(writer, sheet_name=sheet, index=False)

    print(f"{len(df)} unique homogenesous materials written to sheet '{sheet}' in {output_excel}")

In [19]:
extract_HomogeneousMaterial_to_excel(xml_folder = "data/raw/all", output_excel="data/mapping_all.xlsx", sheet = "Sheet4")

26 unique homogenesous materials written to sheet 'Sheet4' in data/mapping_all.xlsx


# load Mapping Once

In [50]:
df_mapping_substances = pd.read_excel(Path("data/mapping_all.xlsx"), sheet_name = "Sheet1") # Load mapping once
df_mapping_homogeneous_materials = pd.read_excel(Path("data/mapping_all.xlsx"), sheet_name = "Sheet4") # idem for homogeneous materials

mapping_dict = {}

# Iterate through substances
for _, row in df_mapping_substances.iterrows():
    substance_name = row["substance name"].strip().lower() 
    # Normalize the substance name for case-insensitive lookup
    act_name = row["ecoinvent activity"]
    location = row["location"]
    mapping_dict[substance_name] = (act_name, location)
    
# Iterate through homogeneous materials
for _, row in df_mapping_homogeneous_materials.iterrows():
    #print(row)
    #if row['homogeneous material'] != nan:
    homogeneous_materials = row["homogeneous material"].strip().lower()
    act_name = row["ecoinvent activity"]
    location = row["location"]
    mapping_dict[homogeneous_materials] = (act_name, location)
    
def get_ecoinvent_activity(name):
    key = name.strip().lower()
    return mapping_dict[key]

In [51]:
# example -> return the corresponding EI activity
# get_ecoinvent_activity("Epoxy Adhesive")

In [52]:
#df_mapping_homogeneous_materials

In [53]:
#mapping_dict

# Parsing

In [66]:
def parse_ipc1752_to_yaml(xml_file: str, output_folder: str):
    """
    Parse a single IPC-1752 XML BOM file into a flattened YAML activity.
    
    Rules:
    - One YAML file per XML (product)
    - Flatten SubProducts → HomogeneousMaterials → Substances
    - Keep unit as mg
    - Assign productref_input_001, _002, etc.
    - + homogeneous materials
    """
    
    # Load XML
    parser = etree.XMLParser(ns_clean=True)
    tree = etree.parse(xml_file, parser)
    
    # Define namespaces
    ns = {'ipc': 'http://webstds.ipc.org/175x/2.0'}
    
    # Get product info
    product_node = tree.find('.//ipc:Product', ns)
    product_id_node = product_node.find('ipc:ProductID', ns)
    product_name = f"{product_id_node.get('itemName')}_{product_id_node.get('version')}"
    total_mass = float((product_id_node.find('ipc:Amount', ns).get('value', 0)) if product_id_node.find('ipc:Amount', ns) is not None else 0)
    
    # Prepare inputs list
    material_inputs = {}
    process_inputs = {}
    counter = 1
    counter_hm = 1
    computed_mass = 0.0
    
    # ------------------------------------------------------
    subproducts = product_node.findall('ipc:SubProduct', ns)
    for sp in subproducts: # iteration in all subproducts
        sp_product_node = sp.find('ipc:ProductID', ns)
        if sp_product_node is not None:
            sp_name = sp_product_node.get('itemName', f"SubProduct_{counter}")
        else:
            sp_name = f"SubProduct_{counter}"
            
        # ------------------------------------------------------
        hmlist = sp.findall('.//ipc:HomogeneousMaterial', ns)
        for hm in hmlist: # iteration in all homogeneous substances
            hm_name = hm.get('name', f"HM_{counter}")

            ref_key_hm = f"process_input_{counter_hm:03d}"
            counter_hm += 1
            
            #ref_key_hm = f"(Homogeneous Material) {hm_name}"
            act_name_hm, location_hm = get_ecoinvent_activity(hm_name)
            
            hm_value_str = hm.get('value') # Try to get value directly from the Substance node
            if hm_value_str is None: # If not present, look for a child <Amount> node
                hm_amount_node = hm.find('ipc:Amount', ns)
                if hm_amount_node is not None:
                    hm_value_str = hm_amount_node.get('value', '0')
                else:
                    hm_value_str = '0'
                    print(f"Error: The value for the homogeneous {hm_name} in file {xml_file.name} is zero")
            hm_value = float(hm_value_str)
            
            
            if act_name_hm == 'no match':
                process_inputs[ref_key_hm] = {
                    'act_name': 'metal working, average for copper product manufacturing',
                    'location': 'RoW',
                    'c_subproduct': sp_name,
                    'c_homogeneous_material': hm_name,
                    'c_substance': 'process',
                    'amount': {
                        'value': 0,
                        'unit': 'mg'
                    }
                }
            else: 
                # uncomment to write yaml as a foreground
                process_inputs[ref_key_hm] = {
                    'act_name': act_name_hm,
                    'location': location_hm,
                    'c_subproduct': sp_name,
                    'c_homogeneous_material': hm_name,
                    'c_substance': 'process',
                    'amount': {
                        'value': hm_value,
                        'unit': 'mg'
                    }
                }

            # ------------------------------------------------------
            subs = hm.findall('.//ipc:Substance', ns) 
            for sub in subs: # iteration in all substances
                sub_name = sub.get('name', 'Unknown')
                substance_id_node = sub.find('ipc:SubstanceID', ns)
                if substance_id_node is not None:
                    cas = substance_id_node.get('identity', 'Unknown')
                else:
                    cas = 'Unknown'
                    
                value_str = sub.get('value') # Try to get value directly from the Substance node
                if value_str is None: # If not present, look for a child <Amount> node
                    amount_node = sub.find('ipc:Amount', ns)
                    if amount_node is not None:
                        value_str = amount_node.get('value', '0')
                    else:
                        value_str = '0'
                        print(f"Error: The value for the substance {sub_name} in file {xml_file.name} is zero")
                
                value = float(value_str)
                
                # Assign productref_input_xxx
                ref_key = f"material_input_{counter:03d}"
                counter += 1
                act_name, location = get_ecoinvent_activity(sub_name)
                
                if act_name == 'no match':
                    material_inputs[ref_key] = {
                        'act_name': 'market for copper, anode',
                        'location': 'GLO',
                        'c_subproduct': sp_name,
                        'c_homogeneous_material': hm_name,
                        'c_substance': sub_name,
                        'amount': {
                            'value': 0,
                            'unit': 'mg'
                        }
                    }
                    
                else: 
                    # In case: 'c_cas': cas,
                    # Uncomment below if foregrounds
                    material_inputs[ref_key] = {
                        'act_name': act_name,
                        'location': location,
                        'c_subproduct': sp_name,
                        'c_homogeneous_material': hm_name,
                        'c_substance': sub_name,
                        'amount': {
                            'value': value,
                            'unit': 'mg'
                        }
                    }
                computed_mass += value
    
    # Mass verification
    mass_check = None
    tolerance = 0.01  # mg
    if abs(computed_mass - total_mass) > tolerance:
        mass_check = f"WARNING: mismatch total_mass={total_mass} vs sum_substances={computed_mass:.6f}"
    
    # Build YAML dict
    # Comment below to generate a foreground instead of custom acts

    # ---- Custom activities ----------------
    yaml_data = {
        'output': {
            'product': product_name,
            'amount': {'value': 1, 'unit': 'unit'}
        },
        #'c_total_mass': total_mass,
        #'c_sum_substances': round(computed_mass, 6)
    }
    #if mass_check:
        #yaml_data['mass_check'] = mass_check
    
    yaml_data['inputs'] = material_inputs | process_inputs
    
    # ---- foreground ----------------
    # yaml_data = material_inputs | process_inputs
    
    
    # Write YAML
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    
    yaml_file = output_folder / f"{product_name}.yaml"
    with open(yaml_file, 'w') as f:
        yaml.dump(yaml_data, f, sort_keys=False)
    
    #print(f"Parsed {xml_file} → {yaml_file}")

In [68]:
from pathlib import Path

# Base paths
RAW_XML_FOLDER = Path("data/raw")
YAML_OUTPUT_FOLDER = Path("data/yaml")

# Target folder containing XML files
xml_folder = RAW_XML_FOLDER / "all"

# Iterate over all XML files
for xml_file in xml_folder.glob("*.xml"):
    parse_ipc1752_to_yaml(xml_file, YAML_OUTPUT_FOLDER)

In [61]:
def extract_component_names(xml_folder):
    ns = {'ipc': 'http://webstds.ipc.org/175x/2.0'}
    components = []

    for xml_file in Path(xml_folder).glob("*.xml"):
        tree = etree.parse(xml_file)
        root = tree.getroot()

        product = root.find(".//ipc:ProductID", ns)

        if product is not None:
            name = product.get("itemName")
            version = product.get("version")

            if name and version:
                comp_name = f"{name}_{version}"
                components.append(comp_name)

    return components

# Packaging meta infos

In [62]:
def get_package_info(component_name, rohs_folder):

    # Remove version suffix (_F, _E, etc.)
    item_name = component_name.split("_")[0]

    xls_path = Path(rohs_folder) / f"{item_name}_eu_rohs.xls"

    try:
        #df = pd.read_excel(xls_path, header=None, engine="openpyxl")
        tables = pd.read_html(xls_path)
        df = tables[1]
        #print(df)
        package_type = df["Package Name"].iloc[0, 0]   # C7
        mass_str = df["Total part weight"].iloc[0, 0]   # D7
        #print(package_type)
        #print(mass_str)
        mass_value = float(mass_str.replace("mg", "").strip())
        match = re.search(r'(\d{1,4})$', package_type)
        pin_count = int(match.group(1)) if match else None

        return package_type, mass_value, pin_count

    except Exception as e:
        print(f"Warning: could not read {xls_path.name}: {e}")
        return "", ""

In [63]:
def generate_foreground_yaml(component_list, rohs_folder, output_file):

    data = {}

    for comp in component_list:

        package_type, mass_value, pin_count = get_package_info(comp, rohs_folder)

        data[comp] = {
            "act_name": comp,
            "c_total_mass": mass_value,
            "c_package_type": package_type,
            "c_pin_count": pin_count, 
            "amount": {
                "value": 1,
                "unit": "unit"
            }
        }

    with open(output_file, "w") as f:
        f.write("# yaml-language-server: $schema=../schemas/foreground.yaml\n\n")
        yaml.dump(data, f, sort_keys=False)

    print(f"Foreground YAML created: {output_file}")

In [64]:
xml_folder = "data/raw/all"
rohs_folder = "data/raw/all_xls"

components = extract_component_names(xml_folder)

generate_foreground_yaml(
    components,
    rohs_folder,
    "data/foreground.yaml"
)

Foreground YAML created: data/foreground.yaml
